<a href="https://www.kaggle.com/code/avikdas567/social-media-extremism-detection-challenge?scriptVersionId=290476414" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/social-media-extremism-detection-challenge/sample_submission.csv
/kaggle/input/social-media-extremism-detection-challenge/train.csv
/kaggle/input/social-media-extremism-detection-challenge/test.csv


In [2]:
!pip install -q pyarrow==22.0.0
!pip install -q transformers accelerate

In [3]:
import pandas as pd
import numpy as np
import re, string, torch
from sklearn.metrics import accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# Load data
train = pd.read_csv("/kaggle/input/social-media-extremism-detection-challenge/train.csv")
test  = pd.read_csv("/kaggle/input/social-media-extremism-detection-challenge/test.csv")

# Preprocess
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"\@\w+|\#", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

train["clean_message"] = train["Original_Message"].apply(clean_text)
test["clean_message"]  = test["Original_Message"].apply(clean_text)

# Map labels
label_map = {"NON_EXTREMIST": 0, "EXTREMIST": 1}
train["labels"] = train["Extremism_Label"].map(label_map)

# Tokenize
MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_enc = tokenizer(list(train["clean_message"]), padding=True, truncation=True, max_length=128)
test_enc  = tokenizer(list(test["clean_message"]),  padding=True, truncation=True, max_length=128)

class SMEDataset(torch.utils.data.Dataset):
    def __init__(self, enc, labels=None):
        self.enc = enc
        self.labels = labels
    def __len__(self):
        return len(self.enc["input_ids"])
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k,v in self.enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[i])
        return item

train_ds = SMEDataset(train_enc, list(train["labels"]))
test_ds  = SMEDataset(test_enc)

# Load model
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Training args
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    learning_rate=3e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=20,
    report_to="none",
    fp16=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=train_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

# Predict test set
pred = trainer.predict(test_ds)
test_preds = np.argmax(pred.predictions, axis=-1)

# Convert back to labels
final_preds = ["EXTREMIST" if p==1 else "NON_EXTREMIST" for p in test_preds]

# Create submission
submission = pd.DataFrame({
    "ID": test["ID"],
    "Extremism_Label": final_preds
})

submission.to_csv("submission.csv", index=False)
print("submission.csv saved!")
submission.head()

2026-01-07 06:20:13.941894: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767766814.132872      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767766814.186726      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767766814.652137      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767766814.652183      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767766814.652186      23 computation_placer.cc:177] computation placer alr

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy
1,0.578400,0.380762,0.830667
2,0.361000,0.210033,0.924000
3,0.222700,0.102716,0.965333
4,0.129800,0.067654,0.980000


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


submission.csv saved!


,ID,Extremism_Label
0,2251,NON_EXTREMIST
1,2252,NON_EXTREMIST
2,2253,NON_EXTREMIST
3,2254,NON_EXTREMIST
4,2255,NON_EXTREMIST
